# MycoCell: Multi-scale Mycoplasma Simulator

A multi-scale cell simulator for JCVI-syn3A (minimal Mycoplasma) built from scratch.

**Architecture:**
- `BiochemNet` — deterministic ODE for metabolism (304 metabolites, 244 reactions)
- Particle simulator — stochastic enzymes with positions
- Spatial hybrid — couples both via voxel grid with Fickian diffusion

**Validated on toy systems:**
- Michaelis-Menten kinetics: 1-4% error
- Reaction-diffusion PDE: 1.4% error
- Mass conservation: <0.005%

**This notebook:**
1. Clones the repo and loads iMB155
2. Tests the architecture at scale (304 species, 244 reactions)
3. Runs a 15-gene essentiality smoke test (honest disclaimer: small sample)
4. Attempts to extend to full Mycoplasma via external data fetches

---

## 1. Setup

In [ ]:
# Setup: clone repo and enter directory (for Colab)
# If you're running locally from a clone, this is a no-op.

import os

if not os.path.exists('mycocell') and not os.path.exists('../mycocell'):
    # Not yet inside the repo — clone it
    !git clone https://github.com/Nikku03/mycocell_sim.git
    os.chdir('mycocell_sim')
elif os.path.basename(os.getcwd()) != 'mycocell_sim' and os.path.exists('mycocell_sim'):
    os.chdir('mycocell_sim')

print('Working directory:', os.getcwd())
print('Contents:', sorted(os.listdir('.')))

In [ ]:
# Install dependencies (most are pre-installed on Colab)
!pip install --quiet cobra scipy numpy matplotlib

In [ ]:
# Import the mycocell package
import sys
sys.path.insert(0, '.')

from mycocell import imb155, kinetics, essentiality
from mycocell.simulator import BiochemNet, SpatialHybrid

import numpy as np
import matplotlib.pyplot as plt
import time

## 2. Load iMB155

In [ ]:
model = imb155.load_model(data_dir='data')

print(f'Loaded iMB155:')
print(f'  Metabolites: {len(model["met_ids"])}')
print(f'  Reactions:   {len(model["rxn_ids"])}')
print(f'  Genes:       {len(model["gene_labels"])}')
print(f'  Stoichiometric matrix shape: {model["S"].shape}')

In [ ]:
# Build kinetic parameter arrays
kin = kinetics.build_rate_arrays(model['rxn_ids'])
print(f'Reactions with measured kinetics: {kin["n_measured"]}')
print(f'Reactions with default kinetics:  {kin["n_default"]}')
print(f'Default Km: {kin["default_km"]} mM')

## 3. Build and run BiochemNet

In [ ]:
net = BiochemNet(
    S=model['S'],
    vmax_f=kin['vmax_f'],
    vmax_r=kin['vmax_r'],
    km_per_rxn=kin['km_per_rxn'],
    met_ids=model['met_ids'],
    default_km=kin['default_km'],
)

print(f'BiochemNet built: {net.n_mets} metabolites, {net.n_rxns} reactions')

In [ ]:
# Integrate from uniform 1 mM starting state
C0 = np.ones(net.n_mets)

t0 = time.perf_counter()
sol = net.integrate(C0, t_max=0.1)  # 100 ms virtual time
elapsed = time.perf_counter() - t0

print(f'Integration: {sol.message}')
print(f'Time steps taken: {sol.t.size}')
print(f'Wall time: {elapsed:.1f}s')
print(f'Final concentration range: [{sol.y[:, -1].min():.3g}, {sol.y[:, -1].max():.3g}] mM')

In [ ]:
# Plot the top 20 most dynamic metabolites
changes = np.abs(sol.y[:, -1] - sol.y[:, 0])
top_idx = np.argsort(-changes)[:20]

plt.figure(figsize=(10, 6))
for i in top_idx:
    plt.plot(sol.t * 1000, sol.y[i], label=model['met_ids'][i])
plt.xlabel('time (ms)')
plt.ylabel('concentration (mM)')
plt.title('Top 20 most dynamic metabolites during simulation')
plt.legend(fontsize=7, loc='center left', bbox_to_anchor=(1, 0.5))
plt.yscale('log')
plt.tight_layout()
plt.show()

## 4. Essentiality smoke test (15 genes)

**Important disclaimer:** this test uses only 15 genes (the subset where we have both iMB155 reaction mapping AND Hutchison 2016 essentiality labels). Any accuracy number has ~±15pp confidence interval. This is a smoke test for the architecture, not a validation of essentiality prediction accuracy.

The full 155-gene mapping requires Breuer 2019 supplementary table S1.

In [ ]:
# Build gene -> reaction indices for the 15 mapped genes
gene_to_rxns = {}
for gene, rxn_name in essentiality.GENE_TO_RXN_NAME.items():
    idx = imb155.find_reaction_index(model['rxn_ids'], rxn_name)
    if idx is not None:
        gene_to_rxns[gene] = [idx]

print(f'Mapped {len(gene_to_rxns)}/{len(essentiality.GENE_TO_RXN_NAME)} genes to reactions')

# Run essentiality evaluation
result = essentiality.evaluate_essentiality(
    net, C0, gene_to_rxns, essentiality.HUTCHISON_LABELS, t_max=0.1)

print(f'\nWT verdict: {result["wt_verdict"]}')
print(f'\n{"gene":20s} {"true":5s} {"verdict":15s} {"correct?":5s}')
print('-' * 50)
for r in result['results']:
    mark = '✓' if r['correct'] else '✗'
    print(f'{r["gene"]:20s} {r["true_label"]:5s} {r["ko_verdict"]:15s} {mark}')

In [ ]:
# Summary statistics (with the n=15 disclaimer)
n_correct = sum(r['correct'] for r in result['results'])
n_total = len(result['results'])
tp = sum(1 for r in result['results'] if r['true_essential'] and r['predicted_essential'])
fp = sum(1 for r in result['results'] if not r['true_essential'] and r['predicted_essential'])
tn = sum(1 for r in result['results'] if not r['true_essential'] and not r['predicted_essential'])
fn = sum(1 for r in result['results'] if r['true_essential'] and not r['predicted_essential'])

print(f'Accuracy: {n_correct}/{n_total} = {n_correct/n_total*100:.1f}%')
print(f'TP={tp} FP={fp} TN={tn} FN={fn}')
print(f'\nNOTE: n={n_total} is not statistically meaningful.')
print(f'Confidence interval is roughly ±15 percentage points.')

## 5. Fetch real data from Luthey-Schulten Lab

The simulator above uses default rate constants for 233 of 244 reactions and uniform 1 mM initial conditions. To get real essentiality predictions, we need the data published with the Thornburg 2022 paper (*Cell*).

**Three files from `Luthey-Schulten-Lab/Minimal_Cell_ComplexFormation/input_data/`:**

1. **`Syn3A_updated.xml`** — Updated SBML model with biomass equation (replaces our iMB155_NoH2O.xml)
2. **`initial_concentrations.xlsx`** — Measured metabolite concentrations (replaces uniform 1 mM)
3. **`kinetic_params.xlsx`** — Measured kinetic constants for all ODE reactions (replaces default kcats)

These are publicly available on GitHub. Running the cell below downloads them into `data/thornburg_2022/`. Integration into the BiochemNet architecture is future work (see README).

In [ ]:
# Download the real Thornburg 2022 data files from Luthey-Schulten Lab repo
import os
import urllib.request

os.makedirs('data/thornburg_2022', exist_ok=True)

BASE = 'https://raw.githubusercontent.com/Luthey-Schulten-Lab/Minimal_Cell_ComplexFormation/main/input_data/'
FILES = [
    'Syn3A_updated.xml',
    'initial_concentrations.xlsx',
    'kinetic_params.xlsx',
]

for fname in FILES:
    url = BASE + fname
    dst = f'data/thornburg_2022/{fname}'
    if os.path.exists(dst):
        print(f'  {fname}: already downloaded ({os.path.getsize(dst):,} bytes)')
        continue
    try:
        print(f'  Fetching {fname}...')
        urllib.request.urlretrieve(url, dst)
        print(f'    Saved: {os.path.getsize(dst):,} bytes')
    except Exception as e:
        print(f'    Failed: {type(e).__name__}: {e}')

print('\nFiles in data/thornburg_2022/:')
for f in sorted(os.listdir('data/thornburg_2022')):
    size = os.path.getsize(f'data/thornburg_2022/{f}')
    print(f'  {f}: {size:,} bytes')

In [ ]:
# Peek at the downloaded data to confirm what's inside
import pandas as pd

# The xlsx files have multiple sheets
print('=== initial_concentrations.xlsx ===')
try:
    xl = pd.ExcelFile('data/thornburg_2022/initial_concentrations.xlsx')
    print(f'Sheets: {xl.sheet_names}')
    # Show the Intracellular Metabolites sheet — this is what we need for initial conditions
    df = pd.read_excel(xl, 'Intracellular Metabolites')
    print(f'\nIntracellular Metabolites: {df.shape[0]} rows x {df.shape[1]} cols')
    print(df.head(10))
except Exception as e:
    print(f'Error: {e}')

print('\n=== kinetic_params.xlsx ===')
try:
    xl = pd.ExcelFile('data/thornburg_2022/kinetic_params.xlsx')
    print(f'Sheets: {xl.sheet_names}')
    # Show Central metabolism sheet — core kinetics
    if 'Central' in xl.sheet_names:
        df = pd.read_excel(xl, 'Central')
        print(f'\nCentral: {df.shape[0]} rows x {df.shape[1]} cols')
        print(df.head(5))
except Exception as e:
    print(f'Error: {e}')

## 6. Spatial hybrid demonstration (optional)

Demonstrates that the particle + voxel architecture produces spatial gradients with appropriate physics. This is separate from the essentiality work above.

In [ ]:
# Spatial hybrid: enzymes in corner, substrate must diffuse in
from mycocell.simulator import SpatialHybrid

# Slow substrate diffusion to force a gradient
sim = SpatialHybrid(
    n_enzymes=50,
    S_total_molecules=1000,
    cell_size_m=1e-6,
    n_voxels_per_side=4,
    enzyme_region=(0, 0.25e-6, 0, 0.25e-6, 0, 0.25e-6),  # small corner
    D_substrate=6.6e-12,  # 100× slower than small-molecule default
    kon=1e8, koff=100.0, kcat=1000.0,
    rng_seed=0,
)

t0 = time.perf_counter()
sim.run(t_max=0.2, dt=5e-5, record_every=200)
print(f'Wall time: {time.perf_counter()-t0:.1f}s')
print(f'Events: {sim.events}')

In [ ]:
# Visualize the substrate field — should show depletion in enzyme corner
S_field = sim.S_field * sim.grid.voxel_volume_L * 6.022e23  # molecules per voxel

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for i, ax in enumerate(axes):
    im = ax.imshow(S_field[:, :, i], origin='lower', cmap='viridis', vmin=0)
    ax.set_title(f'z-slice {i}')
    plt.colorbar(im, ax=ax)
plt.suptitle('Substrate distribution after 200 ms (enzymes in (0,0,0) corner)')
plt.tight_layout()
plt.show()

print(f'Corner voxel (enzymes): {S_field[0,0,0]:.1f} molecules')
print(f'Opposite corner:        {S_field[-1,-1,-1]:.1f} molecules')
print(f'Gradient ratio:         {S_field[0,0,0]/max(S_field[-1,-1,-1], 0.01):.3f}')

## 6b. Integrate Thornburg data into BiochemNet

This section uses our `mycocell.thornburg` adapter to:

1. Parse `kinetic_params.xlsx` (6 metabolism sheets) → ~200 reactions with measured kcat and Km
2. Parse `initial_concentrations.xlsx` → 140 measured metabolite concentrations
3. Rebuild BiochemNet with real data
4. Re-run integration and compare to the uniform-1 mM run above

**Expected:** replacing defaults with measured values should change the dynamics substantially. Whether the simulation is more *biologically correct* depends on how many reactions we cover and whether reaction IDs match up.

In [ ]:
from mycocell.thornburg import (
    load_kinetics, load_initial_concentrations,
    build_rate_arrays_thornburg, build_C0_from_thornburg
)

# 1. Load measured kinetics
print('Loading Thornburg kinetic parameters...')
kin_thornburg = load_kinetics('data/thornburg_2022/kinetic_params.xlsx', verbose=True)

# 2. Load measured initial concentrations
print('\nLoading Thornburg initial concentrations...')
ic_thornburg = load_initial_concentrations(
    'data/thornburg_2022/initial_concentrations.xlsx', verbose=True)

# 3. Build new rate arrays using Thornburg data
print('\nMapping Thornburg reactions onto iMB155...')
kin_new = build_rate_arrays_thornburg(model['rxn_ids'], kin_thornburg, verbose=True)

# 4. Build new C0 using Thornburg concentrations
print('\nBuilding initial concentration vector...')
C0_new = build_C0_from_thornburg(model['met_ids'], ic_thornburg, verbose=True)

In [ ]:
# Build a new BiochemNet with the real data
net_real = BiochemNet(
    S=model['S'],
    vmax_f=kin_new['vmax_f'],
    vmax_r=kin_new['vmax_r'],
    km_per_rxn=kin_new['km_per_rxn'],
    met_ids=model['met_ids'],
    default_km=kin_new['default_km'],
)

print(f'BiochemNet with real data: {net_real.n_mets} mets, {net_real.n_rxns} reactions')
print(f'  Measured reactions: {kin_new["n_measured"]} ({kin_new["n_measured"]/net_real.n_rxns*100:.1f}%)')
print(f'  Default reactions:  {kin_new["n_default"]}')

# Quick integration test — 10 ms to see if it's stable
t0 = time.perf_counter()
sol_real = net_real.integrate(C0_new, t_max=0.01)
elapsed = time.perf_counter() - t0

print(f'\nIntegration (10 ms virtual): {sol_real.message}')
print(f'  Wall time: {elapsed:.1f}s')
print(f'  Time steps: {sol_real.t.size}')

if sol_real.success:
    C_final_real = sol_real.y[:, -1]
    print(f'  Final conc range: [{C_final_real.min():.3e}, {C_final_real.max():.2f}] mM')
    n_dep = int((C_final_real < 0.001).sum())
    n_exp = int((C_final_real > 1000).sum())
    print(f'  Depleted: {n_dep}, exploded: {n_exp}')

In [ ]:
# Re-run the 15-gene essentiality smoke test with REAL data
print('Running essentiality smoke test with Thornburg data...')
print('(~2-3 minutes for 12 knockouts)\n')

# Use same gene-to-reaction mapping as before
gene_to_rxns_real = {}
for gene, rxn_name in essentiality.GENE_TO_RXN_NAME.items():
    idx = imb155.find_reaction_index(model['rxn_ids'], rxn_name)
    if idx is not None:
        gene_to_rxns_real[gene] = [idx]

# Use shorter t_max since C0 is now close to steady-state (not uniform 1mM)
result_real = essentiality.evaluate_essentiality(
    net_real, C0_new, gene_to_rxns_real, essentiality.HUTCHISON_LABELS,
    t_max=0.1)

print(f'WT verdict: {result_real["wt_verdict"]}')
print(f'\n{"gene":20s} {"true":5s} {"verdict":15s} {"correct?":5s}')
print('-' * 50)
for r in result_real['results']:
    mark = '✓' if r['correct'] else '✗'
    print(f'{r["gene"]:20s} {r["true_label"]:5s} {r["ko_verdict"]:15s} {mark}')

n_correct_real = sum(r['correct'] for r in result_real['results'])
n_total_real = len(result_real['results'])
print(f'\nAccuracy with real data: {n_correct_real}/{n_total_real} = '
      f'{n_correct_real/n_total_real*100:.1f}%')
print(f'Baseline (uniform 1mM, mostly defaults): 1/12 = 8.3%')
print(f'Note: still n={n_total_real}, CI is ±15pp — architecture demo, not validation')

## 6c. Split biomass into independent black holes

The original R_BIOMASS lumps all 53 precursors into a single multiplicative rate. This means **one precursor depleting drops the whole biomass rate to zero**, regardless of what actually failed.

**The fix:** split R_BIOMASS into 5-6 independent sub-reactions, one per functional category (amino acids, RNA precursors, DNA precursors, lipids, cofactors, ions). Each sub-reaction has its own kinetics and responds only to its own precursors.

**Testing:** we measure flux through each sub-biomass before and after each knockout. If the sub-reactions discriminate (different KOs hit different categories), the architecture is working. If they all cascade together, we have shared-dependency issues that need diagnosis.

This also loads the upgraded Syn3A_updated.xml (308 mets, 356 rxns) instead of iMB155 (304 mets, 244 rxns).

In [ ]:
from mycocell.syn3a import load_syn3a, find_reaction_index as find_syn3a_rxn

syn3a = load_syn3a('data/thornburg_2022/Syn3A_updated.xml', verbose=True)
print(f"\nSyn3A_updated model loaded.")
print(f"  Metabolites: {len(syn3a['met_ids'])}")
print(f"  Reactions:   {len(syn3a['rxn_ids'])}")
print(f"  Gene products: {len(syn3a['gene_labels'])}")
print(f"  Biomass at idx {syn3a['biomass_rxn_idx']}, exchange at idx {syn3a['biomass_exchange_idx']}")

In [ ]:
from mycocell.blackholes_split import split_biomass_reaction

split_model = split_biomass_reaction(syn3a, verbose=True)

In [ ]:
from mycocell.thornburg import (
    load_kinetics, load_initial_concentrations,
    build_rate_arrays_thornburg, build_C0_from_thornburg,
)
from mycocell.blackholes_split import configure_split_biomass_kinetics

# Load Thornburg kinetic params
thornburg_kin = load_kinetics('data/thornburg_2022/kinetic_params.xlsx', verbose=True)

# Build initial rate arrays for the ORIGINAL reaction set (before split)
kin_base = build_rate_arrays_thornburg(syn3a['rxn_ids'], thornburg_kin, verbose=True)

# Extend the arrays to cover sub-biomass reactions
print('\nConfiguring split-biomass kinetics...')
vmax_f, vmax_r, km_per_rxn = configure_split_biomass_kinetics(
    split_model,
    kin_base['vmax_f'], kin_base['vmax_r'], kin_base['km_per_rxn'],
    biomass_kcat=1.0, biomass_km=0.01, exchange_rate=100.0,
    zero_original=True, verbose=True,
)

# Load initial concentrations
print('\nLoading initial concentrations...')
ic_thornburg = load_initial_concentrations(
    'data/thornburg_2022/initial_concentrations.xlsx', verbose=False)
print(f'  Loaded {len(ic_thornburg)} concentrations')

# Build C0 for the EXTENDED metabolite list (includes sub-biomass pools)
# Sub-biomass pools start at 0 (they're built up from the reactions)
import numpy as np
C0 = build_C0_from_thornburg(split_model['met_ids'], ic_thornburg,
                              default_conc=1.0, verbose=True)

In [ ]:
from mycocell.simulator import BiochemNet
from mycocell.blackholes_split import sub_biomass_fluxes, sub_biomass_summary
import time

net = BiochemNet(
    S=split_model['S'],
    vmax_f=vmax_f, vmax_r=vmax_r,
    km_per_rxn=km_per_rxn,
    met_ids=split_model['met_ids'],
    default_km=kin_base['default_km'],
)
print(f'BiochemNet: {net.n_mets} mets, {net.n_rxns} rxns')

# Run WT with 100s virtual time
print(f'\nWT integration (100s virtual)...')
t0 = time.perf_counter()
sol_wt = net.integrate(C0, t_max=100.0)
print(f'  {sol_wt.message}')
print(f'  Wall time: {time.perf_counter() - t0:.2f}s, steps: {sol_wt.t.size}')

if sol_wt.success:
    wt_fluxes = sub_biomass_fluxes(net, sol_wt, split_model)
    wt_summary = sub_biomass_summary(wt_fluxes, sol_wt)
    
    print(f'\nWT sub-biomass fluxes (mean over last 20% of simulation):')
    print(f'  {"category":20s} {"mean flux (mM/s)":>18s} {"n precursors":>12s}')
    print(f'  {"-" * 52}')
    for cat, stats in wt_summary.items():
        n_prec = len(split_model['category_precursor_indices'][cat])
        print(f'  {cat:20s} {stats["mean_late"]:18.3e} {n_prec:>12d}')
else:
    print('WT simulation failed. Not worth running knockouts.')

In [ ]:
from mycocell import essentiality

# The 15 genes we have Hutchison labels for — map to reactions in Syn3A
# Syn3A gene IDs are G_MMSYN1_XXXX; Hutchison uses JCVISYN3A_XXXX
# We use the GENE_TO_RXN_NAME map from essentiality.py to go via reaction names

# Build gene_to_rxns from Syn3A reaction IDs + Hutchison gene names
syn3a_gene_to_rxns = {}
for hutchison_gene, rxn_short in essentiality.GENE_TO_RXN_NAME.items():
    idx = find_syn3a_rxn(split_model['rxn_ids'], rxn_short)
    if idx is not None:
        syn3a_gene_to_rxns[hutchison_gene] = [idx]

print(f'Mapped {len(syn3a_gene_to_rxns)}/{len(essentiality.GENE_TO_RXN_NAME)} '
      f'Hutchison genes to Syn3A reactions')

# Run knockouts and track sub-biomass fluxes per category
print(f'\nRunning {len(syn3a_gene_to_rxns)} knockouts (~2 min total)...\n')

results = []
for hutchison_gene, rxn_idxs in syn3a_gene_to_rxns.items():
    rxn_short = essentiality.GENE_TO_RXN_NAME[hutchison_gene]
    true_label = essentiality.HUTCHISON_LABELS[hutchison_gene]
    true_essential = essentiality.is_essential(true_label)
    
    ko_net = net.knockout(rxn_idxs)
    t0 = time.perf_counter()
    sol_ko = ko_net.integrate(C0, t_max=100.0)
    elapsed = time.perf_counter() - t0
    
    if sol_ko.success:
        ko_fluxes = sub_biomass_fluxes(ko_net, sol_ko, split_model)
        ko_summary = sub_biomass_summary(ko_fluxes, sol_ko)
    else:
        ko_summary = {cat: {'mean_late': 0.0} for cat in wt_summary}
    
    # Compute fraction per category
    fractions = {}
    for cat in wt_summary:
        wt_rate = wt_summary[cat]['mean_late']
        ko_rate = ko_summary[cat]['mean_late']
        fractions[cat] = ko_rate / max(abs(wt_rate), 1e-12) if wt_rate > 0 else 0.0
    
    # Minimum fraction across categories — this is our viability metric
    min_fraction = min(fractions.values())
    # Essential if ANY category drops below threshold
    threshold = 0.1  # knockout drops any black hole below 10% of WT
    predicted_essential = min_fraction < threshold
    correct = predicted_essential == true_essential
    
    results.append({
        'gene': hutchison_gene,
        'rxn': rxn_short,
        'true_label': true_label,
        'true_essential': true_essential,
        'fractions': fractions,
        'min_fraction': min_fraction,
        'predicted_essential': predicted_essential,
        'correct': correct,
        'solver_success': sol_ko.success,
        'elapsed_s': elapsed,
    })
    
    print(f'  {hutchison_gene} ({rxn_short}): true={true_label}, '
          f'min_frac={min_fraction:.2f}, pred={"E" if predicted_essential else "N"} '
          f'{"✓" if correct else "✗"} ({elapsed:.1f}s)')

# Summary
n_correct = sum(r['correct'] for r in results)
n_total = len(results)
print(f'\n{"=" * 60}')
print(f'Accuracy with split-biomass black holes: {n_correct}/{n_total} = '
      f'{n_correct/n_total*100:.1f}%')
print(f'Baseline (single R_BIOMASS, yesterday): 50%')
print(f'Baseline (uniform 1mM, defaults only): 8.3%')
print(f'Sample size n={n_total}, confidence interval ±15pp')

In [ ]:
# Show the per-category breakdown — which black hole each knockout hits

categories = list(wt_summary.keys())

print(f'\nPer-black-hole impact (fraction of WT throughput):\n')
header = f'  {"gene":20s} {"rxn":8s} {"true":5s} '
header += ' '.join(f'{c[:10]:>10s}' for c in categories)
header += '  pred'
print(header)
print('  ' + '-' * (len(header) - 2))

for r in results:
    row = f'  {r["gene"]:20s} {r["rxn"]:8s} {r["true_label"]:5s} '
    for cat in categories:
        frac = r['fractions'][cat]
        # Color coding: ✗ for <10%, ~ for 10-90%, ✓ for >90%
        if frac < 0.1:
            marker = '✗'
        elif frac < 0.9:
            marker = '~'
        else:
            marker = '✓'
        row += f' {100*frac:>8.1f}% '
    row += f' {"E" if r["predicted_essential"] else "N"}'
    row += f' {"✓" if r["correct"] else "✗"}'
    print(row)

print(f'\nLegend: percent of WT flux for each black hole after the knockout.')
print(f'A clean result would show different knockouts affecting different black holes.')

In [ ]:
# Visualize: heatmap of knockouts × black holes
import matplotlib.pyplot as plt
import numpy as np

gene_labels = [r['gene'].replace('JCVISYN3A_', '') + f' ({r["rxn"]})' for r in results]
category_labels = categories

# Build matrix
M = np.array([[r['fractions'][c] for c in categories] for r in results])

fig, ax = plt.subplots(figsize=(1 + 1.5 * len(categories), 0.3 * len(results) + 2))
im = ax.imshow(M, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
ax.set_xticks(range(len(categories)))
ax.set_xticklabels(category_labels, rotation=30, ha='right')
ax.set_yticks(range(len(results)))
ax.set_yticklabels(gene_labels)

# Annotate cells
for i in range(len(results)):
    for j in range(len(categories)):
        val = M[i, j]
        text = f'{100*val:.0f}'
        color = 'white' if val < 0.3 or val > 0.85 else 'black'
        ax.text(j, i, text, ha='center', va='center', color=color, fontsize=8)

# Mark correct/incorrect predictions on the right
for i, r in enumerate(results):
    ax.text(len(categories) - 0.3, i,
            ('✓' if r['correct'] else '✗') + f' {r["true_label"]}',
            ha='left', va='center',
            color=('green' if r['correct'] else 'red'), fontsize=9)

ax.set_title('Sub-biomass flux after knockout (% of WT)')
plt.colorbar(im, ax=ax, label='Fraction of WT flux')
plt.tight_layout()
plt.show()

In [ ]:
# Save all results to a JSON file for offline analysis / sharing
import json
from datetime import datetime
import os

# Build a serializable output
output = {
    'timestamp': datetime.utcnow().isoformat() + 'Z',
    'model': {
        'source': 'Syn3A_updated.xml',
        'n_metabolites_original': len(syn3a['met_ids']),
        'n_reactions_original': len(syn3a['rxn_ids']),
        'n_metabolites_after_split': len(split_model['met_ids']),
        'n_reactions_after_split': len(split_model['rxn_ids']),
    },
    'kinetics': {
        'source': 'Thornburg 2022 kinetic_params.xlsx',
        'n_measured_reactions': int(kin_base['n_measured']),
        'n_default_reactions': int(kin_base['n_default']),
        'measured_fraction': float(kin_base['n_measured'] / (kin_base['n_measured'] + kin_base['n_default'])),
    },
    'initial_concentrations': {
        'source': 'Thornburg 2022 initial_concentrations.xlsx',
        'n_loaded': len(ic_thornburg),
        'n_metabolites_total': int(len(split_model['met_ids'])),
        'range_mM': [float(min(ic_thornburg.values())), float(max(ic_thornburg.values()))] if ic_thornburg else None,
    },
    'simulation_parameters': {
        't_max_seconds': 100.0,
        'biomass_kcat_per_s': 1.0,
        'biomass_km_mM': 0.01,
        'exchange_rate_per_s': 100.0,
        'essentiality_threshold_fraction': 0.1,
    },
    'black_holes': {
        'categories': list(wt_summary.keys()),
        'n_precursors_per_category': {
            cat: int(len(split_model['category_precursor_indices'][cat]))
            for cat in wt_summary.keys()
        },
        'wt_flux_per_category_mMps': {
            cat: float(stats['mean_late']) for cat, stats in wt_summary.items()
        },
    },
    'wt_simulation': {
        'success': bool(sol_wt.success),
        'n_timesteps': int(sol_wt.t.size),
        'message': str(sol_wt.message),
    },
    'knockouts': [
        {
            'gene': r['gene'],
            'reaction': r['rxn'],
            'hutchison_label': r['true_label'],
            'true_essential': bool(r['true_essential']),
            'fractions_per_blackhole': {
                cat: float(r['fractions'][cat]) for cat in r['fractions']
            },
            'min_fraction': float(r['min_fraction']),
            'predicted_essential': bool(r['predicted_essential']),
            'correct': bool(r['correct']),
            'solver_success': bool(r['solver_success']),
            'elapsed_s': float(r['elapsed_s']),
        }
        for r in results
    ],
    'summary': {
        'n_knockouts_tested': int(len(results)),
        'n_correct': int(sum(r['correct'] for r in results)),
        'accuracy': float(sum(r['correct'] for r in results) / len(results)) if results else 0.0,
        'baselines': {
            'single_biomass_with_thornburg_data': 0.50,
            'single_biomass_uniform_1mM': 0.083,
        },
    },
    'caveats': [
        f'Sample size n={len(results)} — confidence interval ±15pp',
        'Only 15 of 155 iMB155 genes have Hutchison labels mapped',
        'Simulation time 100s virtual; real biological essentiality plays out over minutes',
        'biomass_kcat is not measured — chosen for demonstration purposes',
    ],
}

# Save to a file named with timestamp
ts = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
filename = f'results_blackholes_{ts}.json'
with open(filename, 'w') as f:
    json.dump(output, f, indent=2)

print(f'Saved results to: {filename}')
print(f'  Size: {os.path.getsize(filename):,} bytes')
print(f'  To download from Colab:')
print(f'    from google.colab import files; files.download("{filename}")')
print(f'\nSummary:')
print(f'  Accuracy: {output["summary"]["accuracy"]*100:.1f}% ({output["summary"]["n_correct"]}/{output["summary"]["n_knockouts_tested"]})')
print(f'  Previous baseline: 50%')
print(f'  Change: {(output["summary"]["accuracy"] - 0.50)*100:+.1f} percentage points')

## 6d. Retry with Liebig's law of the minimum

The previous result showed three of eight black holes with near-zero WT flux (amino_acids, other, ions). Investigation suggests this is the **multiplicative Michaelis-Menten collapse problem**: with 20 amino acid precursors, the product of saturation terms goes to nearly zero if even one precursor is slightly low.

**Fix:** replace the multiplicative rate law for sub-biomass reactions with a soft-minimum formulation based on Liebig's law:

$$\text{rate} = \frac{v_{\max}}{1 + \sum_i (C_i^{\text{ref}} / C_i)}$$

Properties:
- If all precursors plentiful: rate ≈ $v_{\max}$
- If one precursor scarce: that term dominates, rate drops proportionally
- Smooth (differentiable everywhere) — solver-friendly
- Responds to the limiting precursor rather than multiplicatively collapsing

We re-run the essentiality test with this rate law and compare.

In [ ]:
from mycocell.biomass_liebig import build_liebig_net_for_split

# Rebuild net using Liebig rate law for sub-biomass reactions
net_liebig = build_liebig_net_for_split(
    split_model, vmax_f, vmax_r, km_per_rxn, C0,
    default_km=kin_base['default_km'],
    ref_scale=0.1,  # rate starts dropping when precursor falls below 10% of initial
)

print(f'Liebig net: {net_liebig.n_mets} mets, {net_liebig.n_rxns} rxns')
print(f'Soft-min reactions: {len(net_liebig.liebig_rxn_indices)}')

# WT run
print(f'\nWT integration with Liebig rate law...')
t0 = time.perf_counter()
sol_wt_L = net_liebig.integrate(C0, t_max=100.0)
print(f'  {sol_wt_L.message}')
print(f'  Wall: {time.perf_counter()-t0:.2f}s, steps: {sol_wt_L.t.size}')

wt_fluxes_L = sub_biomass_fluxes(net_liebig, sol_wt_L, split_model)
wt_summary_L = sub_biomass_summary(wt_fluxes_L, sol_wt_L)

print(f'\nWT sub-biomass fluxes with Liebig law:')
print(f'  {"category":20s} {"old flux (mM/s)":>18s} {"new flux (mM/s)":>18s} {"status":>10s}')
for cat in wt_summary_L:
    old = wt_summary[cat]['mean_late']
    new = wt_summary_L[cat]['mean_late']
    status = 'FIXED' if old < 1e-9 and new > 1e-9 else ('broken' if new < 1e-9 else 'ok')
    print(f'  {cat:20s} {old:18.3e} {new:18.3e} {status:>10s}')

In [ ]:
# Re-run knockouts with Liebig rate law
print(f'Running {len(syn3a_gene_to_rxns)} knockouts with Liebig rate law...\n')

results_L = []
for hutchison_gene, rxn_idxs in syn3a_gene_to_rxns.items():
    rxn_short = essentiality.GENE_TO_RXN_NAME[hutchison_gene]
    true_label = essentiality.HUTCHISON_LABELS[hutchison_gene]
    true_essential = essentiality.is_essential(true_label)
    
    ko_net = net_liebig.knockout(rxn_idxs)
    t0 = time.perf_counter()
    sol_ko = ko_net.integrate(C0, t_max=100.0)
    elapsed = time.perf_counter() - t0
    
    if sol_ko.success:
        ko_fluxes = sub_biomass_fluxes(ko_net, sol_ko, split_model)
        ko_summary = sub_biomass_summary(ko_fluxes, sol_ko)
    else:
        ko_summary = {cat: {'mean_late': 0.0} for cat in wt_summary_L}
    
    fractions = {}
    for cat in wt_summary_L:
        wt_rate = wt_summary_L[cat]['mean_late']
        ko_rate = ko_summary[cat]['mean_late']
        fractions[cat] = ko_rate / max(abs(wt_rate), 1e-12) if wt_rate > 1e-12 else 1.0
    
    # Only use categories that had nonzero WT flux
    valid_fractions = {cat: f for cat, f in fractions.items() if wt_summary_L[cat]['mean_late'] > 1e-9}
    min_fraction = min(valid_fractions.values()) if valid_fractions else 1.0
    
    threshold = 0.1
    predicted_essential = min_fraction < threshold
    correct = predicted_essential == true_essential
    
    results_L.append({
        'gene': hutchison_gene,
        'rxn': rxn_short,
        'true_label': true_label,
        'true_essential': true_essential,
        'fractions': fractions,
        'valid_fractions': valid_fractions,
        'min_fraction': min_fraction,
        'predicted_essential': predicted_essential,
        'correct': correct,
        'solver_success': sol_ko.success,
        'elapsed_s': elapsed,
    })
    
    print(f'  {hutchison_gene} ({rxn_short}): true={true_label}, '
          f'min_frac={min_fraction:.2f}, pred={"E" if predicted_essential else "N"} '
          f'{"✓" if correct else "✗"} ({elapsed:.1f}s)')

n_correct = sum(r['correct'] for r in results_L)
n_total = len(results_L)
print(f'\n{"="*60}')
print(f'Liebig accuracy: {n_correct}/{n_total} = {n_correct/n_total*100:.1f}%')
print(f'Previous (MM, multiplicative):    50%')
print(f'Split-biomass MM (inflated):      92% (with broken black holes)')
print(f'Split-biomass MM (honest, only valid black holes): ~58%')
print(f'{"="*60}')

In [ ]:
# Heatmap for Liebig results
import matplotlib.pyplot as plt
import numpy as np

gene_labels = [r['gene'].replace('JCVISYN3A_', '') + f' ({r["rxn"]})' for r in results_L]
cat_labels = list(wt_summary_L.keys())

M = np.array([[r['fractions'][c] for c in cat_labels] for r in results_L])

fig, ax = plt.subplots(figsize=(1 + 1.5 * len(cat_labels), 0.3 * len(results_L) + 2))
im = ax.imshow(M, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1.2)
ax.set_xticks(range(len(cat_labels)))
ax.set_xticklabels(cat_labels, rotation=30, ha='right')
ax.set_yticks(range(len(results_L)))
ax.set_yticklabels(gene_labels)

# Mark the broken black holes (WT flux < 1e-9) with an X
for j, cat in enumerate(cat_labels):
    if wt_summary_L[cat]['mean_late'] < 1e-9:
        ax.axvspan(j - 0.5, j + 0.5, color='lightgray', alpha=0.5, zorder=0)

for i in range(len(results_L)):
    for j in range(len(cat_labels)):
        val = M[i, j]
        if wt_summary_L[cat_labels[j]]['mean_late'] < 1e-9:
            text = 'N/A'
        else:
            text = f'{100*min(val, 9.99):.0f}'
        color = 'white' if 0.3 > val or val > 0.85 else 'black'
        ax.text(j, i, text, ha='center', va='center', color=color, fontsize=8)

for i, r in enumerate(results_L):
    ax.text(len(cat_labels) - 0.3, i,
            ('✓' if r['correct'] else '✗') + f' {r["true_label"]}',
            ha='left', va='center',
            color=('green' if r['correct'] else 'red'), fontsize=9)

ax.set_title('Sub-biomass flux after knockout (Liebig law). Gray columns = broken.')
plt.colorbar(im, ax=ax, label='Fraction of WT')
plt.tight_layout()
plt.show()

## 7. Summary and next steps

**What this notebook demonstrates:**
- BiochemNet scales to 244-reaction iMB155 and integrates stably
- Spatial hybrid produces physically correct substrate gradients
- With default rate constants and uniform initial conditions, 8% accuracy on 15-gene smoke test (as documented — this is an architecture demo, not a validated result)

**The real data is downloadable from Luthey-Schulten Lab** (Section 5 above). The next step is writing adapters to:

1. Parse `Syn3A_updated.xml` into the same npz format our BiochemNet expects
2. Load measured metabolite concentrations from `initial_concentrations.xlsx` as initial conditions
3. Load measured rate constants from `kinetic_params.xlsx` into our kinetics module
4. Re-run the essentiality smoke test with real data

This adapter work is estimated at 2-3 hours of focused coding and would replace our 8% result with the simulator's real predictive capability.

**For contribution or attribution:** all data files come from Luthey-Schulten Lab's published repositories. See README for citations.